In [1]:
import sys
sys.path.append("..")

In [2]:
# ==========================================
# CELDA 2: IMPORTACIÓN DE LIBRERÍAS
# ==========================================
import openeo
import logging
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import pandas as pd
import numpy as np
from pipeline.ingesta import *
from pipeline.modulo_vpm import *
from pipeline.modulo_fenologico import *
from utils.graficas import *
from config import ROOT
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display, clear_output

# Configurar un logging básico para ver las advertencias o errores de la API de openEO
logging.basicConfig(level=logging.INFO)

print("✅ Entorno preparado. Librerías importadas correctamente.")

2026-07-07 19:23:44.886 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-07 19:23:44.887 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-07 19:23:44.888 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-07 19:23:44.889 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


✅ Entorno preparado. Librerías importadas correctamente.


In [ ]:
#Autenticación con openeo
# openeofed.dataspace.copernicus.eu
# openeo.eodc.eu
print("🔄 Conectando al backend de OpenEO...")
connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()

In [ ]:
fedconnection = openeo.connect("openeofed.dataspace.copernicus.eu").authenticate_oidc()

In [ ]:
job = connection.job('j-26070703330849faa99be189e1c9a831')
logs = job.logs()

with open("job_logs.txt", "w", encoding="utf-8") as f:
    for entry in logs:
        f.write(f"[{entry['time']}] {entry['level']}: {entry['message']}\n")


In [ ]:
# ==========================================
# CARGA DE GEOJSON DE PARCELAS
# ==========================================
import json
import geopandas as gpd
from config import ROOT

# 1. Ruta de tu archivo GeoJSON en Google Drive
ruta_parcelas = ROOT / "data" / "PoligonosMaizPlayitas.geojson"

try:
    gdf_parcelas = gpd.read_file(ruta_parcelas)
    print(f"✅ Archivo cargado con éxito. Total de parcelas: {len(gdf_parcelas)}")
except Exception as e:
    print(f"❌ Error al leer el archivo. Verifica la ruta en tu Drive: {e}")

# 2. Forzar EPSG:4326 estrictamente para el filtro del catálogo de openEO
if gdf_parcelas.crs != "EPSG:4326":
    print("🔄 Reproyectando temporalmente a EPSG:4326 para openEO...")
    gdf_parcelas = gdf_parcelas.to_crs("EPSG:4326")

# Convertir el GeoDataFrame a un diccionario GeoJSON nativo de Python
geojson_openeo = json.loads(gdf_parcelas.to_json())


In [ ]:
# 1. Definir el rango de fechas para el ciclo de maíz (Ejemplo: Campaña de Primera 2025)
temp_ext = ["2025-01-01", "2026-01-01"]

In [ ]:
import numpy as np
import pandas as pd


def extraer_serie_para_sos(
    resultado_preprocesamiento: dict[str, pd.DataFrame],
    id_parcela: str,
    indice: str = "EVI",
) -> tuple[np.ndarray, np.ndarray]:
    """
    Transforma la salida de preprocesar_indices_vpm al formato (serie, fechas)
    que espera detectar_sos, para una parcela y un índice específicos.

    Parámetros
    ----------
    resultado_preprocesamiento : dict[str, pd.DataFrame]
        Salida de preprocesar_indices_vpm. Se espera que cada DataFrame tenga
        índice DatetimeIndex (reindexado diario) y una columna por id_parcela.
    id_parcela : str
        Identificador de la parcela a extraer.
    indice : str, opcional
        "EVI" o "LSWI" (por defecto "EVI").

    Retorna
    -------
    tuple(np.ndarray, np.ndarray)
        (serie, fechas) listos para pasar como argumentos posicionales/keyword
        a detectar_sos. `fechas` son datetime64, ordenados cronológicamente
        ascendente. Pares con NaN en el valor se descartan.

    Lanza
    -----
    KeyError
        Si `indice` no está en resultado_preprocesamiento o `id_parcela` no
        existe como columna.
    ValueError
        Si tras descartar NaN no queda ninguna observación (parcela sin datos
        válidos en el rango solicitado).
    """
    if indice not in resultado_preprocesamiento:
        raise KeyError(
            f"'{indice}' no está en el resultado. Claves disponibles: "
            f"{list(resultado_preprocesamiento.keys())}"
        )

    df = resultado_preprocesamiento[indice]

    if id_parcela not in df.columns:
        raise KeyError(
            f"id_parcela '{id_parcela}' no está en las columnas de '{indice}'."
        )

    col = df[id_parcela].sort_index()

    mascara_validos = col.notna()
    if not mascara_validos.any():
        raise ValueError(
            f"La parcela '{id_parcela}' no tiene observaciones válidas en '{indice}'."
        )

    serie = col.loc[mascara_validos].to_numpy(dtype=float)
    fechas = col.loc[mascara_validos].index.to_numpy(dtype="datetime64[ns]")

    return serie, fechas

In [ ]:
# 2. Cargar únicamente la banda SCL para calcular la máscara morfológica
scl_cube = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent=geojson_openeo,
    temporal_extent=temp_ext,
    bands=["SCL"]
)

# Aplicar el proceso nativo de CDSE para expandir máscaras de nubes y sombras
cloud_mask = scl_cube.process(
    "to_scl_dilation_mask",
    data=scl_cube,
    kernel1_size=21, kernel2_size=59,
    mask1_values=[2, 4, 5, 6, 7],
    mask2_values=[3, 8, 9, 10, 11],
    erosion_kernel_size=3)

In [ ]:
# 1. Cargar la colección acotándola al espacio y tiempo (carga solo una banda para que sea instantáneo)
cube = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent=geojson_openeo,
    temporal_extent=temp_ext,
    bands=["B02"]
)

# 2. Llamar al proceso oficial 'dimension_labels' especificando la dimensión 't' (temporal)
# .execute() obliga al backend a inspeccionar el catálogo y devolver la lista real
fechas_disponibles = cube.dimension_labels(dimension="t").execute()

# 3. Limpiar y ordenar el resultado
# El backend suele devolverlas con hora (ej. '2026-05-17T10:30:00Z'), nos quedamos solo con la fecha YYYY-MM-DD

print(f"Se encontraron {len(fechas_disponibles)} fechas disponibles:")
print(fechas_disponibles)

In [ ]:
print("🛰️ Cargando bandas ópticas para el Modelo VPM (Azul, Rojo, NIR, SWIR)...")

# 3. Cargar las bandas necesarias
datacube_vpm = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent=geojson_openeo,
    temporal_extent=temp_ext,
    bands=["B02", "B04", "B08", "B11"]  # B02 (Azul), B04 (Rojo), B08 (NIR), B11 (SWIR)
)

In [ ]:
# 4. Aplicar la máscara de nubes avanzada y el recorte poligonal estricto
datacube_limpio = datacube_vpm.mask(cloud_mask)
datacube_final = datacube_limpio.mask_polygon(geojson_openeo)

In [ ]:
print("🪄 3. Generando píxeles sintéticos mediante interpolación temporal...")
# Aquí ocurre la magia: openEO rellena cada píxel NaN interpolando con sus vecinos temporales limpios
datacube_interpolado = datacube_final.apply_dimension(
    dimension="t",
    process="array_interpolate_linear"
)

In [ ]:
dfs_vpm_crudos = obtener_datacube_indices_crudo(connection, geojson_openeo, temp_ext[0], temp_ext[1])

In [ ]:
dict_parametros = preprocesar_indices_vpm(dfs_vpm_crudos)

In [ ]:
serie, fechas = extraer_serie_para_sos(dict_parametros, id_parcela="id_0", indice="EVI")

In [ ]:
sos = detectar_sos(
    serie=serie,
    fechas=fechas,
    factor=0.2,
    metodo="seasonal_amplitude",
    ventana_busqueda=(pd.Timestamp("2025-04-01"), pd.Timestamp("2025-04-15")),
)

In [ ]:
print(sos)

In [ ]:
dict_climatico = obtener_datos_climaticos_crudo(fedconnection, geojson_openeo, temp_ext[0], temp_ext[1])

In [ ]:
# =========================================================================
# CELDA: CÁLCULO DE LA PRODUCCIÓN PRIMARIA BRUTA DIARIA CONTINUA (MODELO VPM)
# =========================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("📐 1. Homogeneizando y alineando los horizontes temporales diarios...")
df_t2m = dict_climatico["temperature-mean"]
df_ssrd = dict_climatico["solar-radiation-flux"]

# 1. Asegurar que el clima use el mismo rango temporal diario exacto que la vegetación
df_t2m_diario = df_t2m.reindex(dict_parametros["FPAR"].index, method='nearest')
df_ssrd_diario = df_ssrd.reindex(dict_parametros["FPAR"].index, method='nearest')

# 2. ALINEACIÓN MULTI-PARCELA (Broadcasting de la columna climática regional a todas las parcelas)
# Tomamos la primera columna del clima ('Parcela_1') y la replicamos en la estructura de tus fincas
columnas_parcelas = dict_parametros["FPAR"].columns

df_t2m_aligned = pd.DataFrame(
    np.tile(df_t2m_diario.iloc[:, 0].values, (len(columnas_parcelas), 1)).T,
    index=dict_parametros["FPAR"].index,
    columns=columnas_parcelas
)

df_ssrd_aligned = pd.DataFrame(
    np.tile(df_ssrd_diario.iloc[:, 0].values, (len(columnas_parcelas), 1)).T,
    index=dict_parametros["FPAR"].index,
    columns=columnas_parcelas
)

# --- 3. CONVERSIÓN DE ENERGÍA DIARIA ---
# Convertir J/m²/día a MJ/m²/día (dividido entre 1e6) y extraer el 48% de PAR
df_par = (df_ssrd_aligned / 1e6) * 0.48

# --- 4. CÁLCULO DE T_scalar CON CONDICIONES METEOROLÓGICAS DIARIAS ---
T_min, T_opt, T_max = 10.0, 30.0, 45.0  # Umbrales biológicos C4 (Maíz)

num_t = (df_t2m_aligned - T_min) * (df_t2m_aligned - T_max)
den_t = num_t - ((df_t2m_aligned - T_opt) ** 2)

# Evitar divisiones por cero en el denominador
den_t = den_t.replace(0, np.nan)

df_t_scalar = num_t / den_t
df_t_scalar = df_t_scalar.clip(0.0, 1.0).fillna(0.0)

# Aplicar las restricciones drásticas por frío o estrés térmico por calor
df_t_scalar[df_t2m_aligned < T_min] = 0.0
df_t_scalar[df_t2m_aligned > T_max] = 0.0

# --- 5. CÁLCULO DE LA EFICIENCIA DE USO DE LA LUZ REAL DIARIA (epsilon_g) ---
epsilon_0 = 1.6  # Máxima eficiencia fotosintética teórica para cultivos C4 (Maíz)
df_epsilon = epsilon_0 * df_t_scalar * dict_parametros["W_scalar"]

# --- 6. CÁLCULO DE APAR Y GPP REAL DIARIO ---
df_apar = dict_parametros["FPAR"] * df_par
df_gpp = df_epsilon * df_apar  # Unidad final: g C / m² / día

print("\n✅ ¡Pipeline VPM DIARIO COMPLETADO EXITOSAMENTE!")
print(f"   ✔️ Horizonte temporal continuo evaluado: {df_gpp.shape[0]} días.")
print(f"   ✔️ Temperatura media registrada en la zona: {df_t2m_aligned.mean().mean():.2f} °C")
print(f"   ✔️ GPP Promedio diario del Maíz: {df_gpp.mean().mean():.3f} g C/m²/día")
print(f"   ✔️ GPP Máximo diario alcanzado: {df_gpp.max().max():.2f} g C/m²/día")

# =========================================================================
# GRÁFICA DE CONTROL METODOLÓGICO: EVOLUCIÓN REAL DIARIA
# =========================================================================
plt.figure(figsize=(12, 5))

# Graficar las primeras parcelas para verificar la fluidez diaria continua
max_graficar = min(3, len(df_gpp.columns))
for col in df_gpp.columns[:max_graficar]:
    plt.plot(df_gpp.index, df_gpp[col], linewidth=1.2, alpha=0.6, label=f"Dinámica: {col}")

# Promedio regional masivo en línea negra discontinua (sin marcadores "o" artificiales)
plt.plot(
    df_gpp.index,
    df_gpp.mean(axis=1),
    color='black',
    linewidth=2.5,
    linestyle='--',
    label="Promedio Comayagua (Serie Diaria)"
)

plt.title("Modelo VPM Completo: Dinámica Diaria de la Producción Primaria Bruta (GPP)", fontsize=12, fontweight='bold')
plt.xlabel("Línea de Tiempo del Ciclo de Cultivo (Días)", fontsize=10)
plt.ylabel("GPP ($g\ C / m^2 / día$)", fontsize=10)
plt.xlim(df_gpp.index.min(), df_gpp.index.max())
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# --- El módulo externo recorta tu DataFrame ---
# df_gpp_ciclo = modulo_fenologia.recortar_ciclo(df_gpp, parcela_id)

# --- Invocas tu nueva función reutilizable ---
resultados_produccion = calcular_biomasa_y_rendimiento(
    df_gpp_recortado = df_gpp,
    cue              = 0.56,  # Puedes calibrar estos parámetros por variedad
    harvest_index    = 0.50   # 50% de la planta es grano limpio
)

# Recuperar los objetos resultantes
df_biomasa_dinamica = resultados_produccion["biomasa_acumulada"]
serie_rendimientos  = resultados_produccion["yield_final_tha"]

# Imprimir reporte de rendimiento para tus parcelas en Comayagua
print("🌾 RENDIMIENTOS ESTIMADOS POR PARCELA (t/ha):")
print("-" * 45)
for parcela, valor in serie_rendimientos.items():
    print(f"  🔹 {parcela}: {valor:.2f} toneladas por hectárea")

In [ ]:
df_reporte_final["Area (ha)"] = gdf_parcelas["Area_ha"]
print(df_reporte_final["Area (ha)"])

In [ ]:
import geopandas as gpd

# 1. Cargar tus parcelas y definir explícitamente que nacen en WGS84 (grados)
gdf_parcelas = gpd.GeoDataFrame.from_features(geojson_openeo["features"], crs="EPSG:4326")

# 2. REGLA DE ORO SIG: Para calcular áreas reales en Honduras (Valle de Comayagua),
# debemos pasar de grados (WGS84 / EPSG:4326) a metros usando la proyección UTM local.
# Para la zona central de Honduras se utiliza UTM Zona 16N (EPSG:32616).
gdf_proyectado = gdf_parcelas.to_crs(epsg=32616)

# 3. Calcular el área en metros cuadrados y convertir a Hectáreas (1 ha = 10,000 m²)
gdf_parcelas["Area_ha"] = gdf_proyectado.geometry.area / 10000.0

# 4. Asignar los nombres de las columnas para alinearlos con tus series temporales ("Parcela_1", "Parcela_2"...)
# Nota: Asegúrate de que el orden coincida exactamente con el de tus funciones previas.
gdf_parcelas["id_pipeline"] = [f"id_{i}" for i in range(len(gdf_parcelas))]
gdf_parcelas.set_index("id_pipeline", inplace=True)

# 5. Recuperar la serie de rendimientos que calculó tu función anterior
# (serie_rendimientos es un pd.Series indexado por "Parcela_1", "Parcela_2"...)
df_reporte_final = pd.DataFrame(serie_rendimientos) # Trae la columna "Yield (t/ha)"

# 6. Unir los datos espaciales con las estimaciones numéricas y calcular PRODUCCIÓN
df_reporte_final["Area (ha)"] = gdf_parcelas["Area_ha"]
df_reporte_final["Produccion_Estimada (t)"] = df_reporte_final["Yield (t/ha)"] * df_reporte_final["Area (ha)"]

# --- Visualizar el Reporte de Producción Final ---
print("🌽 REPORTE CONSOLIDADO DE ESTIMACIÓN DE COSECHA DE MAÍZ:")
print("=" * 60)
print(df_reporte_final[["Area (ha)", "Yield (t/ha)", "Produccion_Estimada (t)"]].round(2))
print("=" * 60)
print(f"🚀 PRODUCCIÓN TOTAL ESTIMADA PARA EL PROYECTO: {df_reporte_final['Produccion_Estimada (t)'].sum():.2f} toneladas.")

# Visualización y Validación de Datacubes (No forma parte del pipeline de implementación)

In [ ]:

# =========================================================================
# CONTROL DE CALIDAD: Gráfico de verificación rápido del pipeline continuo
# =========================================================================
df_w_scalar = dict_parametros["W_scalar"]
plt.figure(figsize=(10, 4))

# Graficar el promedio de todas las parcelas a lo largo de la serie diaria
plt.plot(
    df_w_scalar.index,
    df_w_scalar.mean(axis=1),
    color='#2ca02c',
    linewidth=2.5,
    label="Promedio Regional Continuo (Comayagua)"
)

plt.title("Pipeline VPM: Evolución Diaria Continua del Estrés Hídrico ($W_{scalar}$)", fontsize=11, fontweight='bold')
plt.xlabel("Fecha (Resolución Diaria - Whittaker)", fontsize=10)
plt.ylabel("$W_{scalar}$ (1.0 = Sin Estrés Hídrico)", fontsize=10)
plt.grid(True, linestyle="--", alpha=0.5)
plt.xlim(df_w_scalar.index.min(), df_w_scalar.index.max())
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()

In [3]:
graficar_comparativa_whittaker("2023-01-01", "2026-07-06", indice_nombre="EVI", prominencia_min=0.05, distancia_min_dias=70)

✅  Índices cargados desde BD: 1263 fechas × 9 parcelas (2023-01-01 → 2026-07-06).
📈 Suavizando serie temporal para: EVI...
📈 Suavizando serie temporal para: LSWI...

🌾 Calculando FPAR y Factor de Estrés Hídrico Diario (W_scalar)...

✅ Métricas base del VPM calculadas y consolidadas a nivel DIARIO:
   ✔️ Total de Parcelas Procesadas: 9
   ✔️ FPAR Diario (Máx Global): 0.808
   ✔️ W_scalar Diario (Mín/Máx Global): 0.634 / 1.000


interactive(children=(Dropdown(description='Parcela:', options=('id_0', 'id_1', 'id_2', 'id_3', 'id_4', 'id_5'…

In [ ]:
graficar_whittaker_sos("2023-01-01", "2026-07-06", indice_nombre="EVI", prominencia_min=0.05, distancia_min_dias=70, factor_sos=0.2)

✅  Índices cargados desde BD: 1263 fechas × 9 parcelas (2023-01-01 → 2026-07-06).
📈 Suavizando serie temporal para: EVI...
📈 Suavizando serie temporal para: LSWI...

🌾 Calculando FPAR y Factor de Estrés Hídrico Diario (W_scalar)...

✅ Métricas base del VPM calculadas y consolidadas a nivel DIARIO:
   ✔️ Total de Parcelas Procesadas: 9
   ✔️ FPAR Diario (Máx Global): 0.808
   ✔️ W_scalar Diario (Mín/Máx Global): 0.634 / 1.000


interactive(children=(Dropdown(description='🌱 Parcela:', options=('id_0', 'id_1', 'id_2', 'id_3', 'id_4', 'id_…

In [ ]:
graficar_comparativa_whittaker_plotly("2023-01-01", "2026-07-06", indice_nombre="EVI", prominencia_min=0.05, distancia_min_dias=70)

✅  Índices cargados desde BD: 1263 fechas × 9 parcelas (2023-01-01 → 2026-07-06).
📈 Suavizando serie temporal para: EVI...
📈 Suavizando serie temporal para: LSWI...

🌾 Calculando FPAR y Factor de Estrés Hídrico Diario (W_scalar)...

✅ Métricas base del VPM calculadas y consolidadas a nivel DIARIO:
   ✔️ Total de Parcelas Procesadas: 9
   ✔️ FPAR Diario (Máx Global): 0.808
   ✔️ W_scalar Diario (Mín/Máx Global): 0.634 / 1.000


interactive(children=(Dropdown(description='🌱 Parcela:', options=('id_0', 'id_1', 'id_2', 'id_3', 'id_4', 'id_…

In [ ]:
# =========================================================
# CELDA A: DESCARGA DE DATOS DESDE OPENEO
# =========================================================

print("📊 1. Enviando peticiones de reducción zonal a openEO...")
# aggregate_spatial procesa el cubo multitemporal sobre tus parcelas
cube_promedios = datacube_indices.aggregate_spatial(
    geometries=geojson_openeo,
    reducer="mean"
)

print("⏳ Descargando datos reducidos a la memoria local de Colab...")
# Ejecutamos y guardamos el diccionario crudo en una variable global
datos_crudos_openeo = cube_promedios.execute()
print("✅ Datos descargados con éxito. Listos para ser procesados en la siguiente celda.")

In [ ]:
# =========================================================
# CELDA B: PROCESAMIENTO Y GRÁFICA MULTIPARCELA
# =========================================================
import matplotlib.pyplot as plt

# Supongamos que tu función ya devolvió:
# dict_indices = {"EVI": df_evi, "LSWI": df_lswi}
# df_evi = dfs_vpm_crudos["EVI"]
# df_lswi = dfs_vpm_crudos["LSWI"]
df_evi = dfs_vpm_crudos["EVI"]
df_lswi = dfs_vpm_crudos["LSWI"]

num_fechas, num_parcelas = df_evi.shape
print(f"📈 Graficando {num_parcelas} parcelas a lo largo de {num_fechas} fechas.")

# ==========================================
# GRÁFICA 1: EVOLUCIÓN DE EVI POR PARCELA
# ==========================================
plt.figure(figsize=(14, 6))
for col in df_evi.columns:
    plt.plot(
        df_evi.index,
        df_evi[col],
        label=f"Parcela {col}",
        marker="o",
        markersize=4,
        linewidth=1.5,
        alpha=0.8
    )

plt.title("Dinámica Fenológica del Maíz: Evolución de EVI por Parcela", fontsize=14, fontweight="bold")
plt.xlabel("Fecha de Adquisición", fontsize=12)
plt.ylabel("Valor del Índice EVI", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title="Parcelas")
plt.gcf().autofmt_xdate()
plt.tight_layout()
plt.show()

# ==========================================
# GRÁFICA 2: EVOLUCIÓN DE LSWI POR PARCELA
# ==========================================
plt.figure(figsize=(14, 6))
for col in df_lswi.columns:
    plt.plot(
        df_lswi.index,
        df_lswi[col],
        label=f"Parcela {col}",
        marker="s",
        markersize=4,
        linewidth=1.5,
        alpha=0.8
    )

plt.title("Contenido de Humedad en la Cobertura: Evolución de LSWI por Parcela", fontsize=14, fontweight="bold")
plt.xlabel("Fecha de Adquisición", fontsize=12)
plt.ylabel("Valor del Índice LSWI", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title="Parcelas")
plt.gcf().autofmt_xdate()
plt.tight_layout()
plt.show()


In [ ]:
# ==========================================
# GRÁFICA 1: EVOLUCIÓN DE EVI POR PARCELA
# ==========================================
plt.figure(figsize=(14, 6))

# Graficar solo las primeras 3 columnas (parcelas)
for col in df_evi.columns[:3]:
    plt.plot(
        df_evi.index,
        df_evi[col],
        label=f"Parcela {col}",
        marker="o",
        markersize=4,
        linewidth=1.5,
        alpha=0.8
    )

plt.title("Dinámica Fenológica del Maíz: Evolución de EVI (3 primeras parcelas)", fontsize=14, fontweight="bold")
plt.xlabel("Fecha de Adquisición", fontsize=12)
plt.ylabel("Valor del Índice EVI", fontsize=12)
plt.ylim(-1, 1)  # ✅ rango fijo en Y
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title="Parcelas")
plt.gcf().autofmt_xdate()
plt.tight_layout()
plt.show()


In [ ]:
import plotly.express as px

# Graficar solo las primeras 3 parcelas
df_evi_subset = df_evi.iloc[:, :9]

fig = px.line(
    df_evi_subset,
    x=df_evi_subset.index,
    y=df_evi_subset.columns,
    markers=True,
    labels={"value": "EVI", "variable": "Parcela", "index": "Fecha"},
    title="Dinámica Fenológica del Maíz: Evolución de EVI (3 primeras parcelas)"
)

# Ajustar rango del eje Y
fig.update_yaxes(range=[-1, 1])

# Mostrar gráfico interactivo
fig.show()


In [ ]:
dict_vpm = obtener_datacube_indices_crudo(connection, geojson_openeo, temp_ext[0], temp_ext[1])

In [ ]:
# =========================================================
# CELDA B: PROCESAMIENTO Y GRÁFICA MULTIPARCELA (formato dict de DataFrames)
# =========================================================
import matplotlib.pyplot as plt

# Extraer los DataFrames del diccionario
df_evi = dict_vpm["EVI"]
df_lswi = dict_vpm["LSWI"]

# Asegurar que el índice sea datetime
df_evi.index = pd.to_datetime(df_evi.index)
df_lswi.index = pd.to_datetime(df_lswi.index)

# Número de parcelas
num_parcelas = df_evi.shape[1]
print(f"📈 Graficando {num_parcelas} parcelas a lo largo de {df_evi.shape[0]} fechas.")

# ==========================================
# GRÁFICA 1: EVOLUCIÓN DE EVI POR PARCELA
# ==========================================
plt.figure(figsize=(14, 6))
for col in df_evi.columns:
    plt.plot(
        df_evi.index,
        df_evi[col],
        label=col,
        marker="o",
        markersize=4,
        linewidth=1.5,
        alpha=0.8
    )

plt.title("Dinámica Fenológica del Maíz: Evolución de EVI por Parcela", fontsize=14, fontweight="bold")
plt.xlabel("Fecha de Adquisición (2025)", fontsize=12)
plt.ylabel("Valor del Índice EVI", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title="Parcelas")
plt.gcf().autofmt_xdate()
plt.tight_layout()
plt.show()

# ==========================================
# GRÁFICA 2: EVOLUCIÓN DE LSWI POR PARCELA
# ==========================================
plt.figure(figsize=(14, 6))
for col in df_lswi.columns:
    plt.plot(
        df_lswi.index,
        df_lswi[col],
        label=col,
        marker="s",
        markersize=4,
        linewidth=1.5,
        alpha=0.8
    )

plt.title("Contenido de Humedad en la Cobertura: Evolución de LSWI por Parcela", fontsize=14, fontweight="bold")
plt.xlabel("Fecha de Adquisición (2025)", fontsize=12)
plt.ylabel("Valor del Índice LSWI", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title="Parcelas")
plt.gcf().autofmt_xdate()
plt.tight_layout()
plt.show()


In [ ]:
datacube_rgb = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent=geojson_openeo,
    temporal_extent=temp_ext,
    bands=['B04','B03','B02']
)


In [ ]:
ruta_netcdf = descargar_datacube(datacube_rgb,"datacube_rgb.nc")

In [ ]:
visualizar_banda("datacube_final.nc")

In [ ]:
visualizar_banda("datacube_limpio.nc")

In [ ]:
visualizar_rgb(ruta_netcdf)

In [ ]:
import xarray as xr
import numpy as np
import geopandas as gpd
import rioxarray
from shapely.geometry import mapping
import matplotlib.pyplot as plt

ds = xr.open_dataset("cubo_temporal_indices.nc")
evi = ds["EVI"]

# Asignar CRS al xarray (openEO suele entregar en EPSG:32616 o 4326 — ajusta según tu cubo)
evi = evi.rio.write_crs("EPSG:32616")

# Crear una máscara booleana de píxeles dentro de parcelas
# Rasteriza el GeoDataFrame sobre la misma grilla del cubo
mascara_parcelas = evi.isel(t=0).notnull()  # Aproximación: donde hay datos en cualquier fecha

# Forma más precisa: rasterizar el GeoDataFrame directamente
from rasterio.features import geometry_mask
import numpy as np

transform = evi.rio.transform()
ancho = evi.rio.width
alto = evi.rio.height

# gdf_parcelas debe estar en el mismo CRS que el cubo
gdf_reproj = gdf_parcelas.to_crs(evi.rio.crs)

mascara_dentro = ~geometry_mask(
    [mapping(geom) for geom in gdf_reproj.geometry],
    transform=transform,
    invert=False,
    out_shape=(alto, ancho)
)
# mascara_dentro es True donde hay parcela, False en espacio vacío

print(f"Píxeles totales en el cubo:        {mascara_dentro.size:,}")
print(f"Píxeles dentro de parcelas:        {mascara_dentro.sum():,}")
print(f"Cobertura real de parcelas:        {mascara_dentro.mean()*100:.1f}%")
print()

# Ahora calcula NaN rate SOLO dentro de las parcelas
print("Porcentaje de NaN dentro de parcelas por fecha:")
for i, t in enumerate(evi.t.values):
    banda_t = evi.isel(t=i).values          # array 2D numpy
    pixeles_parcela = banda_t[mascara_dentro]
    nan_rate = np.isnan(pixeles_parcela).mean() * 100
    n_validos = (~np.isnan(pixeles_parcela)).sum()
    print(f"  {str(t)[:10]}: {nan_rate:.1f}% NaN  ({n_validos} px válidos)")

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output

def graficar_comparativa_whittaker_plotly(
    fecha_inicio, fecha_fin,
    indice_nombre="EVI",
    distancia_min_dias=90,
    prominencia_min=0.15,
):
    """
    Versión Plotly del gráfico interactivo de validación fenológica.
    Lee desde BD, aplica Whittaker y segmenta ciclos, mostrando
    todo en un gráfico interactivo con dropdown de parcela.
    """
    print(f"\U0001f4c1 Cargando índices desde BD: {fecha_inicio} a {fecha_fin}...")
    dfs_raw = cargar_indices_desde_bd(fecha_inicio, fecha_fin)
    print(f"\U0001f4c8 Aplicando suavizado Whittaker...")
    dict_parametros = preprocesar_indices_vpm(dfs_raw)

    df_crudo = dfs_raw[indice_nombre]
    df_suave = dict_parametros[indice_nombre]
    parcelas = df_crudo.columns.tolist()

    def crear_figura(parcela):
        fig = go.Figure()

        # 1. Serie suavizada diaria (línea continua)
        fig.add_trace(go.Scatter(
            x=df_suave.index,
            y=df_suave[parcela],
            mode='lines',
            name='Serie Diaria Suavizada (Whittaker)',
            line=dict(color='#1f77b4', width=2),
            hovertemplate='%{x|%Y-%m-%d}<br>%{y:.4f}<extra>Whittaker</extra>',
        ))

        # 2. Puntos originales de Sentinel-2
        df_puntos = df_crudo[parcela].dropna()
        fig.add_trace(go.Scatter(
            x=df_puntos.index,
            y=df_puntos.values,
            mode='markers',
            name='Adquisiciones Reales S2 (Válidas)',
            marker=dict(color='#ff7f0e', size=7, line=dict(color='white', width=0.5)),
            hovertemplate='%{x|%Y-%m-%d}<br>%{y:.4f}<extra>S2</extra>',
        ))

        # 3. Líneas verticales de segmentación
        serie_parcela = df_suave[parcela]
        segmentos = segmentar_ciclos(serie_parcela, distancia_min_dias, prominencia_min)

        fechas_etiquetadas = set()
        y_max = df_suave[parcela].max()
        y_min = df_suave[parcela].min()
        rango_y = y_max - y_min
        pos_y_texto = y_max - rango_y * 0.05

        for i, (inicio, fin) in enumerate(segmentos):
            for fecha in (inicio, fin):
                if fecha not in fechas_etiquetadas:
                    fig.add_vline(
                        x=fecha,
                        line=dict(color='#d62728', dash='dot', width=1.5),
                        name='Límites de Ciclo (Valles)' if i == 0 and fecha == inicio else None,
                    )
                    fig.add_annotation(
                        x=fecha,
                        y=pos_y_texto,
                        text=fecha.strftime('%Y-%m-%d'),
                        showarrow=False,
                        font=dict(color='#d62728', size=10),
                        bgcolor='rgba(255,255,255,0.7)',
                        yshift=5,
                    )
                    fechas_etiquetadas.add(fecha)

        fig.update_layout(
            title=dict(
                text=f"Control de Calidad Fenológica y Segmentación: {indice_nombre} en {parcela}",
                font=dict(size=13, weight='bold'),
            ),
            xaxis_title='Fecha del Calendario Agrícola',
            yaxis_title=f'Valor del Índice ({indice_nombre})',
            xaxis=dict(
                range=[df_suave.index.min(), df_suave.index.max()],
                showgrid=True,
                gridcolor='rgba(0,0,0,0.1)',
            ),
            yaxis=dict(showgrid=True, gridcolor='rgba(0,0,0,0.1)'),
            legend=dict(
                x=0.01, y=0.99,
                bgcolor='rgba(255,255,255,0.8)',
                bordercolor='rgba(0,0,0,0)',
            ),
            hovermode='x unified',
            template='plotly_white',
            margin=dict(l=50, r=30, t=60, b=50),
        )

        return fig

    # Widget dropdown
    dropdown = widgets.Dropdown(
        options=parcelas,
        value=parcelas[0],
        description='\U0001f331 Parcela:',
        disabled=False,
        layout=widgets.Layout(width='300px'),
    )

    out = widgets.Output()

    def actualizar(change):
        with out:
            clear_output(wait=True)
            fig = crear_figura(change.new)
            fig.show()

    dropdown.observe(actualizar, names='value')

    display(dropdown, out)
    with out:
        fig = crear_figura(dropdown.value)
        fig.show()
